# Baseline Forecast Evaluation

This notebook evaluates the previous-season-points persistence benchmark using the same test seasons and metric definitions as the regularized models. Returning clubs inherit their prior EPL total; the merged dataset's promoted-team fallback is retained. The baseline is summarized with tables rather than detailed model diagnostics.


## 1. Imports and shared configuration


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 4031
DEVELOPMENT_SEASONS = list(range(2016, 2023))
TEST_SEASONS = list(range(2023, 2026))
SEASON_LABELS = {season: f"{season}/{str(season + 1)[-2:]}" for season in range(2016, 2026)}
TEAMS_TO_PLOT = [
    "Arsenal",
    "Crystal Palace",
    "Manchester City",
    "Chelsea",
    "Liverpool",
    "Tottenham",
]

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


## 2. Load and validate the modeling data


In [2]:
def locate_dataset(filename="epl_merged_data.xlsx"):
    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        candidate = root / "outputs" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate outputs/{filename} from {Path.cwd().resolve()}"
    )


dataset_path = locate_dataset()
all_teams_data = pd.read_excel(dataset_path)

required_columns = {
    "Season", "Team", "Promoted", "Curr_Points", "EOS_Total_Points"
}
missing_columns = sorted(required_columns.difference(all_teams_data.columns))
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

if all_teams_data.duplicated(["Season", "Team"]).any():
    raise ValueError("The dataset contains duplicate Season-Team observations.")

all_teams_data["Remaining_Points"] = (
    all_teams_data["EOS_Total_Points"] - all_teams_data["Curr_Points"]
)

print(f"Loaded: {dataset_path.resolve()}")
print(f"Observations: {len(all_teams_data):,}")
print(f"Seasons: {all_teams_data['Season'].min()}-{all_teams_data['Season'].max()}")


Loaded: C:\Users\micha\epl_results_prediction\outputs\epl_merged_data.xlsx
Observations: 200
Seasons: 2016-2025


## 3. Generate and evaluate baseline predictions


In [3]:
modeling_data = all_teams_data.copy()
development = modeling_data.loc[modeling_data["Season"].isin(DEVELOPMENT_SEASONS)].copy()
test_data = modeling_data.loc[modeling_data["Season"].isin(TEST_SEASONS)].copy()

# The merged dataset stores the promoted-team fallback in Prev_Total_Points.
if test_data["Prev_Total_Points"].isna().any():
    raise ValueError("Prev_Total_Points contains missing test values.")

test_results = test_data[
    ["Season", "Team", "Promoted", "Curr_Points", "Remaining_Points", "EOS_Total_Points"]
].copy()
test_results["Predicted_Final_Points"] = test_data["Prev_Total_Points"].to_numpy()
test_results["Predicted_Remaining_Points"] = (
    test_results["Predicted_Final_Points"] - test_results["Curr_Points"]
)
test_results["Final_Points_Residual"] = (
    test_results["EOS_Total_Points"] - test_results["Predicted_Final_Points"]
)
test_results["Absolute_Final_Points_Error"] = test_results["Final_Points_Residual"].abs()
test_results["Season_Label"] = test_results["Season"].map(SEASON_LABELS)

test_mse = mean_squared_error(test_results["EOS_Total_Points"], test_results["Predicted_Final_Points"])
test_mae = mean_absolute_error(test_results["EOS_Total_Points"], test_results["Predicted_Final_Points"])
baseline_results = {
    "Label": "Previous-Season Points Baseline",
    "Experiment": "Baseline",
    "Model": "Previous-season persistence",
    "Features": ["Prev_Total_Points"],
    "Best_Parameters": {},
    "Validation_Metrics": pd.DataFrame(),
    "Validation_MSE": np.nan,
    "Validation_MAE": np.nan,
    "Test_Results": test_results,
    "Test_MSE": test_mse,
    "Test_MAE": test_mae,
    "Test_RMSE": np.sqrt(test_mse),
    "Coefficient_Table": pd.DataFrame(
        {"Feature": ["Prev_Total_Points"], "Standardized_Coefficient": [np.nan], "Absolute_Coefficient": [np.nan], "Selected": [True]}
    ),
}

season_metrics = []
for season, group in test_results.groupby("Season"):
    mse = mean_squared_error(group["EOS_Total_Points"], group["Predicted_Final_Points"])
    season_metrics.append(
        {
            "Season": SEASON_LABELS[season],
            "Observations": len(group),
            "MSE": mse,
            "MAE": mean_absolute_error(group["EOS_Total_Points"], group["Predicted_Final_Points"]),
            "RMSE": np.sqrt(mse),
        }
    )

print(f"Test MSE: {test_mse:.3f}")
print(f"Test MAE: {test_mae:.3f}")
print(f"Test RMSE: {np.sqrt(test_mse):.3f}")
display(pd.DataFrame(season_metrics))


Test MSE: 266.803
Test MAE: 12.753
Test RMSE: 16.334


,Season,Observations,MSE,MAE,RMSE
0,2023/24,20,242.963,12.548,15.587
1,2024/25,20,346.873,14.895,18.625
2,2025/26,20,210.573,10.817,14.511
